# LeetCode 1641 — Count Sorted Vowel Strings bằng Python

Link đề: https://leetcode.com/problems/count-sorted-vowel-strings/ · Độ khó: **Medium** · Chủ đề: Math, Dynamic Programming, Combinatorics

Skeleton LeetCode:

```python
class Solution:
    def countVowelStrings(self, n: int) -> int:
        ...
```

Notebook đi từ **sinh hết mọi xâu** rồi lọc → **quay lui** chỉ sinh xâu hợp lệ → **đệ quy có nhớ** → **quy hoạch động tổng tiền tố** → **công thức tổ hợp** "chia kẹo Euler" `C(n+4, 4)` chạy `O(1)`.

## 1. Mục tiêu học

1. Hiểu điều kiện "xâu đã sắp xếp theo thứ tự từ điển" ⇔ mỗi ký tự **không nhỏ hơn** ký tự đứng trước (dãy **không giảm**).
2. Viết được brute force bằng `itertools.product` và quay lui có cắt nhánh, thấy rõ vì sao chúng chậm.
3. Thiết kế trạng thái quy hoạch động `dp[len][last]` và tối ưu bằng **tổng tiền tố**, bộ nhớ `O(1)` (5 ô).
4. Chứng minh công thức tổ hợp `C(n + 4, 4)` bằng kỹ thuật **sao và vạch (stars and bars)**, và tính nó an toàn bằng số nguyên.
5. Đối chiếu nhiều cách với nhau trên toàn bộ miền `1 ≤ n ≤ 50` (và brute force ở `n` nhỏ).

## 2. Đề bài & ví dụ

Cho số nguyên `n`. Hãy trả về số lượng xâu độ dài `n` chỉ gồm các nguyên âm `a, e, i, o, u` và **đã sắp xếp theo thứ tự từ điển**.

Xâu `s` được gọi là đã sắp xếp nếu với mọi chỉ số hợp lệ `i`, `s[i]` **bằng hoặc đứng trước** `s[i+1]` trong bảng chữ cái.

**Ví dụ 1:** `n = 1` → `5` (`"a","e","i","o","u"`).

**Ví dụ 2:** `n = 2` → `15`: `aa, ae, ai, ao, au, ee, ei, eo, eu, ii, io, iu, oo, ou, uu`. Chú ý `"ea"` **không** hợp lệ vì `e` đứng sau `a`.

**Ví dụ 3:** `n = 33` → `66045`.

**Ràng buộc:** `1 ≤ n ≤ 50`.

In [1]:
import itertools
import math
import random
import time
from functools import lru_cache, cache

VOWELS = "aeiou"
examples = [(1, 5), (2, 15), (33, 66045)]

def is_sorted_str(s):
    return all(s[i] <= s[i + 1] for i in range(len(s) - 1))

n = 2
valid = [''.join(p) for p in itertools.product(VOWELS, repeat=n) if is_sorted_str(''.join(p))]
print(f"n={n}: {len(valid)} xâu ->", valid)
print("'ea' hợp lệ?", is_sorted_str("ea"), "| 'ae' hợp lệ?", is_sorted_str("ae"))

n=2: 15 xâu -> ['aa', 'ae', 'ai', 'ao', 'au', 'ee', 'ei', 'eo', 'eu', 'ii', 'io', 'iu', 'oo', 'ou', 'uu']
'ea' hợp lệ? False | 'ae' hợp lệ? True


## 3. Kiến thức nền

### 3.1 So sánh ký tự trong Python

Ký tự được so sánh theo mã Unicode: `'a' < 'e' < 'i' < 'o' < 'u'`. Ta cũng có thể đổi nguyên âm thành **chỉ số** `0..4` để làm việc với số cho gọn.

In [2]:
print('a' < 'e' < 'i' < 'o' < 'u')
print([ord(c) for c in VOWELS])
idx = {c: k for k, c in enumerate(VOWELS)}
print(idx, "| 'io' ->", [idx[c] for c in "io"])

True
[97, 101, 105, 111, 117]
{'a': 0, 'e': 1, 'i': 2, 'o': 3, 'u': 4} | 'io' -> [2, 3]


### 3.2 `itertools.product` — tích Descartes

`product(VOWELS, repeat=n)` sinh **mọi** bộ `n` ký tự (có lặp, có thứ tự): `5ⁿ` bộ. Với `n = 50` đó là `≈ 8.9·10³⁴` — không thể duyệt hết.

In [3]:
print(list(itertools.product("ab", repeat=3)))
for n in [1, 2, 5, 10, 50]:
    print(f"5^{n} = {5 ** n}")

[('a', 'a', 'a'), ('a', 'a', 'b'), ('a', 'b', 'a'), ('a', 'b', 'b'), ('b', 'a', 'a'), ('b', 'a', 'b'), ('b', 'b', 'a'), ('b', 'b', 'b')]
5^1 = 5
5^2 = 25
5^5 = 3125
5^10 = 9765625
5^50 = 88817841970012523233890533447265625


### 3.3 `itertools.combinations_with_replacement`

Sinh các **tổ hợp lặp** theo thứ tự không giảm — chính xác là các xâu đã sắp xếp! Số lượng là `C(k + n − 1, n)` với `k` phần tử, chọn `n` lần.

In [4]:
cwr = [''.join(c) for c in itertools.combinations_with_replacement(VOWELS, 2)]
print(len(cwr), cwr)

15 ['aa', 'ae', 'ai', 'ao', 'au', 'ee', 'ei', 'eo', 'eu', 'ii', 'io', 'iu', 'oo', 'ou', 'uu']


### 3.4 `functools.cache` (ghi nhớ kết quả hàm)

`@cache` (Python ≥ 3.9) hoặc `@lru_cache(maxsize=None)` lưu kết quả theo tham số, gọi lại cùng tham số thì trả ngay. Biến đệ quy "nổ" số lần gọi thành chỉ "số trạng thái × chi phí mỗi trạng thái".

In [5]:
calls = 0
@cache
def fib(k):
    global calls
    calls += 1
    return k if k < 2 else fib(k - 1) + fib(k - 2)

print("fib(60) =", fib(60), "| số lần thân hàm chạy:", calls)

fib(60) = 1548008755920 | số lần thân hàm chạy: 61


### 3.5 `math.comb` — hệ số nhị thức

`math.comb(N, K)` = `N! / (K! (N−K)!)`, tính bằng **số nguyên chính xác**. Nếu tự tính, hãy dùng phép chia nguyên `//` **sau khi** nhân xong, để không bị lẻ.

In [6]:
print(math.comb(6, 4), math.comb(37, 4))
N = 37
print("tự tính:", N * (N - 1) * (N - 2) * (N - 3) // 24)
print("chia số thực:", N * (N - 1) * (N - 2) * (N - 3) / 24, "<- kiểu float, không phải int")

15 66045
tự tính: 66045
chia số thực: 66045.0 <- kiểu float, không phải int


## 4. Ý tưởng nền tảng / trực giác

Một xâu đã sắp xếp được **xác định hoàn toàn** bởi việc mỗi nguyên âm xuất hiện **bao nhiêu lần** — vì thứ tự đã bị "ép" là `a…ae…ei…io…ou…u`.

Ví dụ `n = 3`: xâu `"aeu"` ↔ `(số a, số e, số i, số o, số u) = (1, 1, 0, 0, 1)`; xâu `"iii"` ↔ `(0, 0, 3, 0, 0)`.

Vậy câu hỏi trở thành: **có bao nhiêu bộ 5 số tự nhiên `(x_a, x_e, x_i, x_o, x_u)` có tổng bằng `n`?**

Tính tay theo ký tự **đầu tiên** (hay cuối cùng) cũng cho một lối nghĩ khác. Gọi `cnt[L][v]` = số xâu độ dài `L` **kết thúc** bằng nguyên âm `v`:
- `L = 1`: mỗi nguyên âm 1 xâu ⇒ `[1, 1, 1, 1, 1]`.
- `L = 2`: xâu kết thúc bằng `v` = ghép `v` vào sau một xâu độ dài 1 kết thúc bằng nguyên âm `≤ v` ⇒ `[1, 2, 3, 4, 5]`, tổng `15`.
- `L = 3`: lấy tổng tiền tố của hàng trước ⇒ `[1, 3, 6, 10, 15]`, tổng `35`.

In [7]:
row = [1] * 5
print("L=1:", row, "tổng", sum(row))
for L in range(2, 6):
    row = list(itertools.accumulate(row))
    print(f"L={L}:", row, "tổng", sum(row))

L=1: [1, 1, 1, 1, 1] tổng 5
L=2: [1, 2, 3, 4, 5] tổng 15
L=3: [1, 3, 6, 10, 15] tổng 35
L=4: [1, 4, 10, 20, 35] tổng 70
L=5: [1, 5, 15, 35, 70] tổng 126


Liệt kê tay toàn bộ `35` xâu với `n = 3`, nhóm theo **ký tự đầu**. Để ý: nhóm bắt đầu bằng `a` có `15` xâu (= đáp án của `n = 2` trên 5 chữ), nhóm `e` có `10` (= `n = 2` trên 4 chữ `e,i,o,u`), … — một cách nhìn khác của cùng phép đệ quy.

In [8]:
all3 = [''.join(c) for c in itertools.combinations_with_replacement(VOWELS, 3)]
for v in VOWELS:
    group = [s for s in all3 if s[0] == v]
    print(f"bắt đầu bằng {v}: {len(group):>2} xâu -> {' '.join(group)}")
print("Tổng:", len(all3))

print("\nTam giác Pascal (8 hàng đầu):")
for r in range(8):
    print("  " + " ".join(f"{math.comb(r, c):>3}" for c in range(r + 1)))

bắt đầu bằng a: 15 xâu -> aaa aae aai aao aau aee aei aeo aeu aii aio aiu aoo aou auu
bắt đầu bằng e: 10 xâu -> eee eei eeo eeu eii eio eiu eoo eou euu
bắt đầu bằng i:  6 xâu -> iii iio iiu ioo iou iuu
bắt đầu bằng o:  3 xâu -> ooo oou ouu
bắt đầu bằng u:  1 xâu -> uuu
Tổng: 35

Tam giác Pascal (8 hàng đầu):
    1
    1   1
    1   2   1
    1   3   3   1
    1   4   6   4   1
    1   5  10  10   5   1
    1   6  15  20  15   6   1
    1   7  21  35  35  21   7   1


Các hàng này chính là các đường chéo của **tam giác Pascal** (1,1,1… / 1,2,3… / 1,3,6,10… / số tứ diện…). Đây là gợi ý mạnh rằng đáp án có dạng **hệ số nhị thức**.

## 5. Cách 1 — Brute Force / baseline: sinh hết `5ⁿ` xâu rồi lọc

In [9]:
def count_brute(n):
    cnt = 0
    for tup in itertools.product(range(5), repeat=n):
        if all(tup[i] <= tup[i + 1] for i in range(n - 1)):
            cnt += 1
    return cnt

for n in range(1, 8):
    t0 = time.perf_counter()
    r = count_brute(n)
    print(f"n={n}: {r:>4}   ({(time.perf_counter() - t0) * 1000:.1f} ms, đã duyệt {5 ** n} bộ)")

n=1:    5   (0.0 ms, đã duyệt 5 bộ)
n=2:   15   (0.0 ms, đã duyệt 25 bộ)
n=3:   35   (0.0 ms, đã duyệt 125 bộ)
n=4:   70   (0.2 ms, đã duyệt 625 bộ)
n=5:  126   (0.8 ms, đã duyệt 3125 bộ)
n=6:  210   (3.8 ms, đã duyệt 15625 bộ)
n=7:  330   (19.0 ms, đã duyệt 78125 bộ)


**Độ phức tạp:** `O(5ⁿ · n)` thời gian, `O(n)` bộ nhớ. Mỗi lần tăng `n` thêm 1, thời gian **gấp 5**. `n = 12` đã là `2.4·10⁸` bộ — hết thời gian.

**Nút thắt:** đa số bộ sinh ra là **không hợp lệ** (với `n = 7`: `330` hợp lệ trên `78125`). Ta nên **không sinh** chúng ngay từ đầu.

## 6. Ý tưởng thuật toán / bước đột phá

1. **Cắt nhánh:** khi xây xâu từ trái sang phải, ký tự tiếp theo chỉ được chọn từ `last..4`. Chỉ sinh xâu hợp lệ ⇒ số lá = đáp án (vẫn có thể rất lớn: `316 251` khi `n = 50`).
2. **Trạng thái lặp lại:** số cách hoàn thành xâu chỉ phụ thuộc **còn bao nhiêu vị trí** và **ký tự nhỏ nhất được phép**. ⇒ `dfs(i, j)`, chỉ có `n × 5` trạng thái ⇒ ghi nhớ.
3. **Truy hồi tổng tiền tố:** `f[L][v] = Σ_{u ≤ v} f[L−1][u]` — **nếu** biết hàng trước **thì** hàng sau là tổng tiền tố của nó; cập nhật tại chỗ, chỉ 5 ô.
4. **Đếm trực tiếp:** đáp án = số nghiệm nguyên không âm của `x₁ + … + x₅ = n` = `C(n + 4, 4)` (sao và vạch).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Quay lui chỉ sinh xâu hợp lệ

In [10]:
def count_backtrack(n):
    count = 0
    def go(pos, last):
        nonlocal count
        if pos == n:
            count += 1
            return
        for v in range(last, 5):        # chỉ chọn nguyên âm >= nguyên âm trước
            go(pos + 1, v)
    go(0, 0)
    return count

for n in [1, 2, 3, 10, 20]:
    t0 = time.perf_counter()
    r = count_backtrack(n)
    print(f"n={n}: {r}  ({(time.perf_counter() - t0) * 1000:.1f} ms)")

n=1: 5  (0.0 ms)
n=2: 15  (0.0 ms)
n=3: 35  (0.0 ms)
n=10: 1001  (0.2 ms)
n=20: 10626  (3.4 ms)


**Độ phức tạp:** số nút trên cây đệ quy ≈ tổng số tiền tố hợp lệ = `Σ_{L ≤ n} C(L+4, 4)` = `C(n+5, 5)` ⇒ `O(n⁵)`. Với `n = 50` khoảng `3.5·10⁶` nút — chạy được nhưng chậm (vài giây trong Python). Độ sâu đệ quy chỉ `n ≤ 50`, an toàn.

Code nộp LeetCode cho cách này (chậm nhưng qua được với `n ≤ 50` trên nhiều ngôn ngữ; với Python có thể sát giới hạn — chỉ để tham khảo):

In [11]:
class SolutionBacktrack:
    def countVowelStrings(self, n: int) -> int:
        def go(pos: int, last: int) -> int:
            if pos == n:
                return 1
            return sum(go(pos + 1, v) for v in range(last, 5))
        return go(0, 0)

print([SolutionBacktrack().countVowelStrings(n) for n in [1, 2, 8]])

[5, 15, 495]


### 7.2 Cách 3 — Đệ quy có nhớ `dfs(i, j)`

`dfs(i, j)` = số cách điền các vị trí `i, i+1, …, n−1` khi ký tự nhỏ nhất được phép là nguyên âm `j`.
- Nếu `i == n`: đã điền xong ⇒ `1` cách.
- Ngược lại: thử ký tự `k` từ `j` đến `4`, cộng `dfs(i + 1, k)`.

Chỉ có `(n + 1) × 5` trạng thái, mỗi trạng thái tốn `O(5)` ⇒ `O(25 n)`.

In [12]:
def count_memo(n):
    @cache
    def dfs(i, j):
        if i == n:
            return 1
        return sum(dfs(i + 1, k) for k in range(j, 5))
    return dfs(0, 0)

class SolutionMemo:
    def countVowelStrings(self, n: int) -> int:
        @cache
        def dfs(i: int, j: int) -> int:
            if i == n:
                return 1
            return sum(dfs(i + 1, k) for k in range(j, 5))
        return dfs(0, 0)

for n, exp in examples:
    print(n, count_memo(n), SolutionMemo().countVowelStrings(n), "(kỳ vọng", exp, ")")

1 5 5 (kỳ vọng 5 )
2 15 15 (kỳ vọng 15 )
33 66045 66045 (kỳ vọng 66045 )


**Lưu ý:** đặt `@cache` **bên trong** method để mỗi lần gọi với `n` khác nhau có bộ nhớ đệm riêng. Nếu đặt hàm cache ở ngoài mà phụ thuộc `n` toàn cục, sẽ trả kết quả cũ sai.

### 7.3 Cách 4 — Quy hoạch động bảng + tổng tiền tố (bộ nhớ `O(1)`)

`f[v]` = số xâu độ dài hiện tại **kết thúc** bằng nguyên âm `v`. Từ độ dài `L` lên `L+1`: `f_new[v] = f[0] + f[1] + … + f[v]` (thêm `v` vào cuối xâu có ký tự cuối `≤ v`). Cập nhật **tại chỗ** từ trái sang phải với một biến cộng dồn `s`.

In [13]:
def count_dp_table(n):
    """Bản bảng đầy đủ dp[L][v], dễ hiểu."""
    dp = [[0] * 5 for _ in range(n + 1)]
    dp[1] = [1] * 5
    for L in range(2, n + 1):
        for v in range(5):
            dp[L][v] = sum(dp[L - 1][u] for u in range(v + 1))
    return sum(dp[n])

def count_dp(n):
    """Bản tối ưu: 5 ô, tổng tiền tố tại chỗ."""
    f = [1] * 5
    for _ in range(n - 1):
        s = 0
        for v in range(5):
            s += f[v]
            f[v] = s
    return sum(f)

for n, exp in examples:
    print(n, count_dp_table(n), count_dp(n), "(kỳ vọng", exp, ")")

1 5 5 (kỳ vọng 5 )
2 15 15 (kỳ vọng 15 )
33 66045 66045 (kỳ vọng 66045 )


In [14]:
class SolutionDP:
    def countVowelStrings(self, n: int) -> int:
        f = [1] * 5
        for _ in range(n - 1):
            s = 0
            for v in range(5):
                s += f[v]
                f[v] = s
        return sum(f)

print([SolutionDP().countVowelStrings(n) for n, _ in examples])

[5, 15, 66045]


**Độ phức tạp:** `O(5 n)` = `O(n)` thời gian, `O(1)` bộ nhớ (bản bảng: `O(n)` bộ nhớ).

### 7.4 Cách 5 — Công thức tổ hợp `C(n + 4, 4)`

**Sao và vạch:** biểu diễn xâu bằng `n` ngôi sao (các ký tự) và `4` vạch ngăn chia thành 5 ngăn `a | e | i | o | u`. Ví dụ `n = 3`, xâu `"aeu"` ↔ `★|★|||★`. Mỗi cách đặt 4 vạch giữa `n + 4` vị trí cho **đúng một** xâu và ngược lại ⇒ đáp án = `C(n + 4, 4)`.

Khai triển: `C(n+4, 4) = (n+1)(n+2)(n+3)(n+4) / 24`. Tích 4 số nguyên liên tiếp luôn chia hết cho `4! = 24`, nên dùng `//` là chính xác.

In [15]:
def stars_and_bars(n, s):
    """Đổi xâu đã sắp xếp thành chuỗi sao/vạch."""
    parts = [s.count(v) * '★' for v in VOWELS]
    return '|'.join(parts)

for s in ["aeu", "iii", "aaa", "ouu"]:
    print(f"{s} -> {stars_and_bars(3, s)}")

def count_formula(n):
    return (n + 1) * (n + 2) * (n + 3) * (n + 4) // 24

for n, exp in examples:
    print(n, count_formula(n), math.comb(n + 4, 4), "(kỳ vọng", exp, ")")

aeu -> ★|★|||★
iii -> ||★★★||
aaa -> ★★★||||
ouu -> |||★|★★
1 5 5 (kỳ vọng 5 )
2 15 15 (kỳ vọng 15 )
33 66045 66045 (kỳ vọng 66045 )


**Độ phức tạp:** `O(1)` thời gian và bộ nhớ.

## 8. TRACE từng bước

### 8.1 Trace cách 4 (DP tổng tiền tố) với `n = 4`

Bảng mong đợi (`f[v]` = số xâu kết thúc bằng `v`):

| Độ dài | a | e | i | o | u | tổng |
|---|---|---|---|---|---|---|
| 1 | 1 | 1 | 1 | 1 | 1 | 5 |
| 2 | 1 | 2 | 3 | 4 | 5 | 15 |
| 3 | 1 | 3 | 6 | 10 | 15 | 35 |
| 4 | 1 | 4 | 10 | 20 | 35 | 70 |

In [16]:
def trace_dp(n):
    f = [1] * 5
    print(f"độ dài 1: f={f} tổng={sum(f)}")
    for L in range(2, n + 1):
        s = 0
        for v in range(5):
            s += f[v]
            print(f"   L={L} v={VOWELS[v]}: s += f_cũ[{v}] -> s={s}, gán f[{v}]={s}")
            f[v] = s
        print(f"độ dài {L}: f={f} tổng={sum(f)}")
    return sum(f)

trace_dp(4)

độ dài 1: f=[1, 1, 1, 1, 1] tổng=5
   L=2 v=a: s += f_cũ[0] -> s=1, gán f[0]=1
   L=2 v=e: s += f_cũ[1] -> s=2, gán f[1]=2
   L=2 v=i: s += f_cũ[2] -> s=3, gán f[2]=3
   L=2 v=o: s += f_cũ[3] -> s=4, gán f[3]=4
   L=2 v=u: s += f_cũ[4] -> s=5, gán f[4]=5
độ dài 2: f=[1, 2, 3, 4, 5] tổng=15
   L=3 v=a: s += f_cũ[0] -> s=1, gán f[0]=1
   L=3 v=e: s += f_cũ[1] -> s=3, gán f[1]=3
   L=3 v=i: s += f_cũ[2] -> s=6, gán f[2]=6
   L=3 v=o: s += f_cũ[3] -> s=10, gán f[3]=10
   L=3 v=u: s += f_cũ[4] -> s=15, gán f[4]=15
độ dài 3: f=[1, 3, 6, 10, 15] tổng=35
   L=4 v=a: s += f_cũ[0] -> s=1, gán f[0]=1
   L=4 v=e: s += f_cũ[1] -> s=4, gán f[1]=4
   L=4 v=i: s += f_cũ[2] -> s=10, gán f[2]=10
   L=4 v=o: s += f_cũ[3] -> s=20, gán f[3]=20
   L=4 v=u: s += f_cũ[4] -> s=35, gán f[4]=35
độ dài 4: f=[1, 4, 10, 20, 35] tổng=70


70

### 8.2 Trace cách 3 (đệ quy có nhớ) với `n = 2`

In mỗi lần **thân hàm thực sự chạy** (không tính các lần lấy từ cache) để thấy số trạng thái nhỏ thế nào.

In [17]:
def trace_memo(n):
    memo = {}
    def dfs(i, j, depth=0):
        if (i, j) in memo:
            print("   " * depth + f"dfs({i},{j}) lấy từ memo = {memo[(i, j)]}")
            return memo[(i, j)]
        if i == n:
            res = 1
        else:
            res = sum(dfs(i + 1, k, depth + 1) for k in range(j, 5))
        memo[(i, j)] = res
        print("   " * depth + f"dfs({i},{j}) = {res}")
        return res
    ans = dfs(0, 0)
    print("Số trạng thái đã tính:", len(memo))
    return ans

trace_memo(2)

      dfs(2,0) = 1
      dfs(2,1) = 1
      dfs(2,2) = 1
      dfs(2,3) = 1
      dfs(2,4) = 1
   dfs(1,0) = 5
      dfs(2,1) lấy từ memo = 1
      dfs(2,2) lấy từ memo = 1
      dfs(2,3) lấy từ memo = 1
      dfs(2,4) lấy từ memo = 1
   dfs(1,1) = 4
      dfs(2,2) lấy từ memo = 1
      dfs(2,3) lấy từ memo = 1
      dfs(2,4) lấy từ memo = 1
   dfs(1,2) = 3
      dfs(2,3) lấy từ memo = 1
      dfs(2,4) lấy từ memo = 1
   dfs(1,3) = 2
      dfs(2,4) lấy từ memo = 1
   dfs(1,4) = 1
dfs(0,0) = 15
Số trạng thái đã tính: 11


15

### 8.3 Trace cây quay lui (cách 2) với `n = 2`

Mỗi dòng là một nút của cây: tiền tố hiện tại và các lựa chọn hợp lệ cho vị trí kế tiếp. Lá (đủ độ dài) được đánh dấu `✔`.

In [18]:
def trace_backtrack(n):
    leaves = 0
    def go(prefix, last):
        nonlocal leaves
        if len(prefix) == n:
            leaves += 1
            print("    " * len(prefix) + f"'{prefix}' ✔ (lá thứ {leaves})")
            return
        choices = VOWELS[last:]
        print("    " * len(prefix) + f"'{prefix}' -> được chọn: {choices}")
        for v in range(last, 5):
            go(prefix + VOWELS[v], v)
    go("", 0)
    return leaves

trace_backtrack(2)

'' -> được chọn: aeiou
    'a' -> được chọn: aeiou
        'aa' ✔ (lá thứ 1)
        'ae' ✔ (lá thứ 2)
        'ai' ✔ (lá thứ 3)
        'ao' ✔ (lá thứ 4)
        'au' ✔ (lá thứ 5)
    'e' -> được chọn: eiou
        'ee' ✔ (lá thứ 6)
        'ei' ✔ (lá thứ 7)
        'eo' ✔ (lá thứ 8)
        'eu' ✔ (lá thứ 9)
    'i' -> được chọn: iou
        'ii' ✔ (lá thứ 10)
        'io' ✔ (lá thứ 11)
        'iu' ✔ (lá thứ 12)
    'o' -> được chọn: ou
        'oo' ✔ (lá thứ 13)
        'ou' ✔ (lá thứ 14)
    'u' -> được chọn: u
        'uu' ✔ (lá thứ 15)


15

## 9. Code nộp LeetCode

Lời giải chính: công thức tổ hợp, `O(1)`. (Có thể thay bằng `return comb(n + 4, 4)` với `from math import comb`.)

In [19]:
class Solution:
    def countVowelStrings(self, n: int) -> int:
        return (n + 1) * (n + 2) * (n + 3) * (n + 4) // 24

for n, exp in examples:
    assert Solution().countVowelStrings(n) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Song ánh xâu đã sắp xếp ↔ bộ đếm:** ánh xạ `s ↦ (x_a, x_e, x_i, x_o, x_u)` (số lần xuất hiện mỗi nguyên âm). Nó là **đơn ánh** trên tập xâu đã sắp xếp vì một xâu không giảm được khôi phục duy nhất bằng `'a'*x_a + 'e'*x_e + …`. Nó là **toàn ánh** lên tập nghiệm của `x_a + … + x_u = n, x ≥ 0` vì xâu khôi phục như trên luôn có độ dài `n` và không giảm.

**Sao và vạch:** mỗi nghiệm ↔ một dãy `n + 4` ký hiệu gồm `n` sao và `4` vạch (`x_a` sao, vạch, `x_e` sao, vạch, …). Chọn vị trí cho 4 vạch: `C(n + 4, 4)` cách.

**DP (cách 4):** bất biến "sau `L − 1` vòng, `f[v]` là số xâu độ dài `L` đã sắp xếp kết thúc bằng `v`". Đúng với `L = 1`. Một xâu độ dài `L+1` kết thúc bằng `v` ⇔ xâu độ dài `L` kết thúc bằng `u ≤ v`, nối thêm `v` (phân hoạch theo `u`) ⇒ `f_new[v] = Σ_{u≤v} f[u]`. Trong vòng cập nhật tại chỗ, khi xử lý `v` thì `s` đang bằng `Σ_{u≤v} f_cũ[u]` vì ta cộng `f[v]` **trước khi** ghi đè nó.

**Memo (cách 3):** `dfs(i, j)` đếm đúng theo quy nạp ngược trên `i`: các cách điền vị trí `i` là `k ∈ [j, 4]`, sau đó phần còn lại phải `≥ k` — đúng là `dfs(i+1, k)`; các nhánh `k` khác nhau rời nhau.

Kiểm chứng song ánh bằng máy với `n = 4`: ánh xạ mỗi xâu đã sắp xếp sang bộ đếm, kiểm tra không trùng và phủ hết tập nghiệm của `x₁ + … + x₅ = 4`.

In [20]:
n = 4
strings = [''.join(c) for c in itertools.combinations_with_replacement(VOWELS, n)]
vectors = [tuple(s.count(v) for v in VOWELS) for s in strings]
solutions = [x for x in itertools.product(range(n + 1), repeat=5) if sum(x) == n]
print("số xâu:", len(strings), "| số bộ đếm khác nhau:", len(set(vectors)), "| số nghiệm:", len(solutions))
assert set(vectors) == set(solutions) and len(set(vectors)) == len(strings) == math.comb(n + 4, 4)
rebuilt = [''.join(VOWELS[k] * x[k] for k in range(5)) for x in solutions]
assert sorted(rebuilt) == sorted(strings)
print("Song ánh đúng với n = 4")

số xâu: 70 | số bộ đếm khác nhau: 70 | số nghiệm: 70
Song ánh đúng với n = 4


## 11. Phân tích Time / Space Complexity

| Cách | Time | Space | `n = 50` chạy được? |
|---|---|---|---|
| 1. Sinh `5ⁿ` | `O(5ⁿ · n)` | `O(n)` | không |
| 2. Quay lui | `O(C(n+5, 5))` ≈ `O(n⁵/120)` | `O(n)` stack | được, chậm (~giây) |
| 3. Memo | `O(25 n)` | `O(5 n)` | được |
| 4. DP tiền tố | `O(5 n)` | `O(1)` | được |
| 5. Công thức | `O(1)` | `O(1)` | được |

Tổng quát với `k` chữ cái thay vì 5: DP `O(k·n)`, công thức `C(n + k − 1, k − 1)`.

In [21]:
for name, fn in [("backtrack", count_backtrack), ("memo", count_memo), ("dp", count_dp), ("formula", count_formula)]:
    n = 30 if name == "backtrack" else 50
    t0 = time.perf_counter(); r = fn(n); dt = time.perf_counter() - t0
    print(f"{name:<9} n={n}: {r:>8}  {dt * 1000:8.2f} ms")

backtrack n=30:    46376     30.99 ms
memo      n=50:   316251      0.17 ms
dp        n=50:   316251      0.01 ms
formula   n=50:   316251      0.00 ms


## 12. Bộ test

Miền đầu vào rất nhỏ (`1 ≤ n ≤ 50`) nên ta kiểm **toàn bộ** miền cho các cách nhanh, brute force cho `n ≤ 7`, và kiểm thêm tính chất: tổng `dp` khớp `combinations_with_replacement`.

In [22]:
for n, exp in examples:
    assert count_memo(n) == count_dp(n) == count_dp_table(n) == count_formula(n) == exp
    assert SolutionMemo().countVowelStrings(n) == SolutionDP().countVowelStrings(n) == Solution().countVowelStrings(n) == exp
print("Ví dụ đề: OK")

for n in range(1, 8):
    b = count_brute(n)
    assert b == count_backtrack(n) == count_memo(n) == count_dp(n) == count_formula(n), n
    assert b == sum(1 for _ in itertools.combinations_with_replacement(VOWELS, n))
print("n = 1..7 so với brute force: OK")

for n in range(1, 51):
    v = Solution().countVowelStrings(n)
    assert v == count_dp(n) == count_memo(n) == math.comb(n + 4, 4) == SolutionDP().countVowelStrings(n)
    assert isinstance(v, int)
print("Toàn miền n = 1..50: OK | n=50 ->", Solution().countVowelStrings(50))

Ví dụ đề: OK
n = 1..7 so với brute force: OK
Toàn miền n = 1..50: OK | n=50 -> 316251


In [23]:
# Kiểm tra ngẫu nhiên (seed cố định) quay lui so với công thức cho n vừa phải
rnd = random.Random(1641)
for _ in range(15):
    n = rnd.randint(1, 14)
    assert count_backtrack(n) == count_formula(n), n
# Tính chất: số xâu đã sắp xếp với k chữ cái = C(n+k-1, k-1)
def count_general_dp(n, k):
    f = [1] * k
    for _ in range(n - 1):
        s = 0
        for v in range(k):
            s += f[v]; f[v] = s
    return sum(f)
for k in range(1, 8):
    for n in range(1, 15):
        assert count_general_dp(n, k) == math.comb(n + k - 1, k - 1)
print("Ngẫu nhiên + tổng quát hóa k chữ cái: OK")

Ngẫu nhiên + tổng quát hóa k chữ cái: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Đếm xâu **tăng nghiêm ngặt** thay vì **không giảm**

In [24]:
def count_bug_strict(n):
    @cache
    def dfs(i, j):
        if i == n:
            return 1
        return sum(dfs(i + 1, k) for k in range(j + 1, 5))   # SAI: không cho lặp ký tự
    return sum(dfs(1, k) for k in range(5))

print("n=2 sai:", count_bug_strict(2), "| đúng:", Solution().countVowelStrings(2))

n=2 sai: 10 | đúng: 15


**Vì sao sai:** đề cho phép `s[i]` **bằng** `s[i+1]` (`"aa"` hợp lệ). `range(j + 1, 5)` loại các xâu có ký tự lặp — chỉ còn `C(5, n)`. **Sửa:** `range(j, 5)`.

### Lỗi 2 — Lặp DP `n` lần thay vì `n − 1` lần

In [25]:
def count_bug_loops(n):
    f = [1] * 5                 # f đã ứng với độ dài 1
    for _ in range(n):          # SAI: phải là n - 1
        s = 0
        for v in range(5):
            s += f[v]; f[v] = s
    return sum(f)

print("n=1 sai:", count_bug_loops(1), "| đúng:", Solution().countVowelStrings(1))

n=1 sai: 15 | đúng: 5


**Vì sao sai:** `f = [1]*5` đã là trạng thái độ dài 1; mỗi vòng tăng độ dài thêm 1 ⇒ cần `n − 1` vòng. **Sửa:** `range(n - 1)`, hoặc khởi tạo `f` ứng với độ dài 0 là `[1, 0, 0, 0, 0]` rồi lặp `n` lần.

### Lỗi 3 — Nhầm công thức tổ hợp và chia số thực

In [26]:
def count_bug_comb(n):
    return math.comb(n + 5, 5)                  # SAI: đó là số tiền tố (độ dài <= n), không phải số xâu độ dài n

def count_bug_float(n):
    return (n + 1) * (n + 2) * (n + 3) * (n + 4) / 24   # SAI kiểu: float

print("comb(n+5,5) với n=2:", count_bug_comb(2), "| đúng:", Solution().countVowelStrings(2))
v = count_bug_float(33)
print("bản chia /:", v, type(v).__name__, "| đúng:", Solution().countVowelStrings(33), type(Solution().countVowelStrings(33)).__name__)

comb(n+5,5) với n=2: 21 | đúng: 15
bản chia /: 66045.0 float | đúng: 66045 int


**Vì sao sai:** `n` sao và `5 − 1 = 4` vạch ⇒ `C(n + 4, 4)`. Nhầm số vạch thành 5 cho `C(n+5, 5)` (= số xâu độ dài `≤ n` kể cả rỗng). Còn `/` trả `float` — LeetCode yêu cầu `int`; với số lớn hơn float còn **mất chính xác**. **Sửa:** `math.comb(n + 4, 4)` hoặc `//`.

### Lỗi 4 — Đặt cache ở ngoài, phụ thuộc biến toàn cục `n`

In [27]:
N_GLOBAL = 0

@cache
def dfs_global(i, j):
    if i == N_GLOBAL:
        return 1
    return sum(dfs_global(i + 1, k) for k in range(j, 5))

def count_bug_cache(n):
    global N_GLOBAL
    N_GLOBAL = n                 # SAI: cache vẫn giữ kết quả tính với n cũ
    return dfs_global(0, 0)

print("lần 1, n=2:", count_bug_cache(2))
print("lần 2, n=3:", count_bug_cache(3), "| đúng:", Solution().countVowelStrings(3))

lần 1, n=2: 15
lần 2, n=3: 15 | đúng: 35


**Vì sao sai:** `dfs_global(0, 0)` đã được cache từ lần gọi đầu (với `n = 2`), lần sau trả lại `15` dù `n` đã đổi. Trên LeetCode, nhiều test case chạy trong **cùng một tiến trình** nên lỗi này rất thật. **Sửa:** định nghĩa hàm `@cache` **bên trong** method (như `SolutionMemo`) hoặc đưa `n` vào tham số.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đếm dãy **không giảm** độ dài `n` trên bảng `k` giá trị" ⇒ **tổ hợp lặp** `C(n + k − 1, k − 1)`.
- Thấy các hàng DP là **tổng tiền tố** của hàng trước (`1,1,1` → `1,2,3` → `1,3,6`) ⇒ nghi ngay **tam giác Pascal / hệ số nhị thức**.
- Bất cứ khi nào "thứ tự đã bị ép" (sắp xếp, không giảm), chỉ còn **số lượng mỗi loại** quan trọng ⇒ sao và vạch.
- Khi chưa nghĩ ra công thức: brute force `n` nhỏ → in dãy `5, 15, 35, 70, 126` → tra/so với `math.comb`.
- Dùng `math.comb` để tránh lỗi chia; luôn trả `int`.

## 15. Biến thể của bài

1. **Thay 5 nguyên âm bằng `k` chữ cái bất kỳ:** `C(n + k − 1, k − 1)`.
2. **Tăng nghiêm ngặt:** chọn `n` chữ khác nhau từ `k` ⇒ `C(k, n)`.
3. **Mỗi chữ dùng tối đa `c` lần:** không còn công thức gọn; DP `dp[v][len]` với giới hạn, hoặc bao hàm–loại trừ.
4. **LeetCode 1220 — Count Vowels Permutation:** quy tắc chữ nào được đứng sau chữ nào (đồ thị chuyển trạng thái), DP theo ký tự cuối, có modulo.
5. **Đếm xâu không giảm có tổng "giá trị" cố định** (gán `a=1, e=2, …`): DP hai chiều theo (độ dài, tổng).
6. **In xâu đã sắp xếp thứ `k`** theo thứ tự từ điển: dùng bảng đếm để "nhảy" từng ký tự (giống LeetCode 60 — Permutation Sequence).

In [28]:
def kth_sorted_vowel_string(n, k):
    """Xâu đã sắp xếp thứ k (1-indexed) theo thứ tự từ điển, dùng công thức đếm."""
    def ways(rem, j):                # số cách điền rem vị trí, ký tự >= j
        return math.comb(rem + (5 - j) - 1, (5 - j) - 1) if rem > 0 else 1
    res, j = [], 0
    for pos in range(n):
        for c in range(j, 5):
            w = ways(n - pos - 1, c)
            if k <= w:
                res.append(VOWELS[c]); j = c; break
            k -= w
    return ''.join(res)

all3 = [''.join(c) for c in itertools.combinations_with_replacement(VOWELS, 3)]
assert all(kth_sorted_vowel_string(3, i + 1) == s for i, s in enumerate(all3))
print("Xâu thứ 1, 10, 35 với n=3:", [kth_sorted_vowel_string(3, k) for k in (1, 10, 35)])

Xâu thứ 1, 10, 35 với n=3: ['aaa', 'aii', 'uuu']


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Sinh `5ⁿ` | product + lọc | `O(5ⁿ n)` | `O(n)` | kiểm tra `n ≤ 7` |
| 2. Quay lui | chỉ chọn ký tự `≥` ký tự trước | `O(n⁵)` | `O(n)` | khi cần **liệt kê** xâu |
| 3. Memo `dfs(i,j)` | ghi nhớ trạng thái | `O(25 n)` | `O(5 n)` | khuôn mẫu chung cho DP đếm |
| 4. DP tiền tố | `f[v] = Σ_{u≤v} f[u]` | `O(5 n)` | `O(1)` | khi có thêm ràng buộc, không có công thức |
| 5. Công thức | sao và vạch | `O(1)` | `O(1)` | nộp bài, nhanh nhất |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `Solution().countVowelStrings(3)` bằng bao nhiêu? Và có bao nhiêu xâu độ dài 3 **bắt đầu** bằng `'i'`?

**Bài 2 (tính tay):** Viết chuỗi sao–vạch cho các xâu `"aaee"`, `"eiou"`, `"uuuu"` (`n = 4`), rồi viết ngược xâu cho chuỗi `"|★★||★★|"`.

**Bài 3 (điền TODO):** Hoàn thiện `count_strict(n)` đếm xâu nguyên âm độ dài `n` **tăng nghiêm ngặt**, bằng DP (không dùng `math.comb`), rồi tự kiểm với `C(5, n)`.

**Bài 4 (tự viết lại):** Viết `count_sorted_k(n, k)` cho bảng chữ cái `k` ký tự bằng DP tổng tiền tố, và một bản bằng công thức; so hai bản với `1 ≤ n ≤ 30`, `1 ≤ k ≤ 10`.

**Bài 5 (mini challenge):** Viết `count_sorted_limited(n, c)`: số xâu nguyên âm đã sắp xếp độ dài `n` mà **mỗi nguyên âm xuất hiện tối đa `c` lần**. Kiểm với brute force cho `n ≤ 7`.

In [29]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def count_strict(n):
    # TODO: f[v] = số xâu tăng nghiêm ngặt kết thúc bằng v; f_new[v] = sum(f[u] for u < v)
    pass

# Bài 4 — TODO
def count_sorted_k(n, k):
    # TODO: tổng quát hóa count_dp với k ô
    pass

# Bài 5 — TODO
def count_sorted_limited(n, c):
    # TODO: dp[j][L] = số cách dùng j nguyên âm đầu để tạo độ dài L, mỗi nguyên âm 0..c lần
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1**
- Mức 1: `C(7, 4)`.
- Mức 2: Bắt đầu bằng `'i'` ⇒ 2 ký tự còn lại chọn từ `{i, o, u}`, không giảm.
- Mức 3: `C(2 + 2, 2) = 6`.

**Bài 2**
- Mức 1: Thứ tự ngăn: `a | e | i | o | u`.
- Mức 2: `"aaee"` → `★★|★★|||`.
- Mức 3: `"|★★||★★|"` → 0 `a`, 2 `e`, 0 `i`, 2 `o`, 0 `u` ⇒ `"eeoo"`.

**Bài 3**
- Mức 1: Giống `count_dp` nhưng không cộng `f[v]` của chính `v`.
- Mức 2: Dùng tổng tiền tố **loại trừ**: cộng `s` vào trước rồi mới tăng `s`.
- Mức 3: Khi `n > 5` đáp án là `0`.

**Bài 4**
- Mức 1: Thay `5` bằng `k`.
- Mức 2: Công thức `C(n + k − 1, k − 1)`.
- Mức 3: `k = 1` ⇒ luôn 1 xâu.

**Bài 5**
- Mức 1: Không còn sao–vạch tự do; mỗi ngăn có tối đa `c` sao.
- Mức 2: DP theo từng nguyên âm: `new[L] = Σ_{t=0..min(c, L)} old[L − t]`.
- Mức 3: Khởi tạo `old = [1, 0, 0, …]` (độ dài 0), lặp 5 lần, trả `old[n]`.

## 19. Đáp án tham khảo cho bài tập

In [30]:
# Bài 1
print("Bài 1:", Solution().countVowelStrings(3), "| bắt đầu bằng 'i':",
      sum(1 for c in itertools.combinations_with_replacement(VOWELS, 3) if c[0] == 'i'))
assert Solution().countVowelStrings(3) == 35
assert sum(1 for c in itertools.combinations_with_replacement(VOWELS, 3) if c[0] == 'i') == math.comb(4, 2) == 6

# Bài 2
for s in ["aaee", "eiou", "uuuu"]:
    print(" ", s, "->", stars_and_bars(4, s))
def from_stars(sb):
    return ''.join(VOWELS[k] * part.count('★') for k, part in enumerate(sb.split('|')))
print("  |★★||★★| ->", from_stars("|★★||★★|"))
assert from_stars("|★★||★★|") == "eeoo"
assert all(from_stars(stars_and_bars(4, s)) == s for s in ["aaee", "eiou", "uuuu"])

Bài 1: 35 | bắt đầu bằng 'i': 6
  aaee -> ★★|★★|||
  eiou -> |★|★|★|★
  uuuu -> ||||★★★★
  |★★||★★| -> eeoo


In [31]:
# Bài 3
def count_strict_sol(n):
    f = [1] * 5
    for _ in range(n - 1):
        s = 0
        for v in range(5):
            s, f[v] = s + f[v], s        # f_new[v] = tổng f_cũ[u] với u < v
    return sum(f)

for n in range(1, 9):
    brute = sum(1 for tup in itertools.product(range(5), repeat=n) if all(tup[i] < tup[i + 1] for i in range(n - 1))) if n <= 6 else 0
    assert count_strict_sol(n) == math.comb(5, n) == (brute if n <= 6 else 0)
print("Bài 3 OK:", [count_strict_sol(n) for n in range(1, 8)])

# Bài 4
def count_sorted_k_sol(n, k):
    f = [1] * k
    for _ in range(n - 1):
        s = 0
        for v in range(k):
            s += f[v]; f[v] = s
    return sum(f)

for n in range(1, 31):
    for k in range(1, 11):
        assert count_sorted_k_sol(n, k) == math.comb(n + k - 1, k - 1)
print("Bài 4 OK")

Bài 3 OK: [5, 10, 10, 5, 1, 0, 0]
Bài 4 OK


In [32]:
# Bài 5
def count_sorted_limited_sol(n, c):
    dp = [1] + [0] * n           # dp[L]: số cách với các nguyên âm đã xét
    for _ in range(5):
        new = [0] * (n + 1)
        for L in range(n + 1):
            for t in range(0, min(c, L) + 1):
                new[L] += dp[L - t]
        dp = new
    return dp[n]

def count_sorted_limited_brute(n, c):
    return sum(1 for comb_ in itertools.combinations_with_replacement(range(5), n)
               if max(comb_.count(v) for v in range(5)) <= c)

for n in range(1, 8):
    for c in range(0, 8):
        assert count_sorted_limited_sol(n, c) == count_sorted_limited_brute(n, c), (n, c)
assert count_sorted_limited_sol(33, 33) == 66045
print("Bài 5 OK | n=4, c=1:", count_sorted_limited_sol(4, 1), "| n=6, c=2:", count_sorted_limited_sol(6, 2))

Bài 5 OK | n=4, c=1: 5 | n=6, c=2: 45


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template DP "dãy không giảm trên k giá trị":**

```python
f = [1] * k                  # độ dài 1
for _ in range(n - 1):
    s = 0
    for v in range(k):
        s += f[v]            # cộng TRƯỚC khi ghi đè
        f[v] = s
return sum(f)
```

**Template công thức:** `math.comb(n + k - 1, k - 1)` (với bài này `k = 5`: `(n+1)(n+2)(n+3)(n+4) // 24`).

**Template memo:** `@cache def dfs(i, j): return 1 if i == n else sum(dfs(i+1, x) for x in range(j, k))`, đặt **bên trong** method.

**Checklist:**
- [ ] Điều kiện là **không giảm** (`≤`), không phải tăng nghiêm ngặt?
- [ ] Số vòng DP đúng (`n − 1` nếu khởi tạo ở độ dài 1)?
- [ ] Công thức `C(n + 4, 4)` — 4 vạch cho 5 ngăn?
- [ ] Dùng `//` hoặc `math.comb`, trả về `int`?
- [ ] Cache không bị "dính" giữa các test?
- [ ] Đã kiểm toàn miền `n = 1..50` và brute force `n ≤ 7`?
- [ ] Chữ ký đúng: `def countVowelStrings(self, n: int) -> int`?